# Transparency and Explainability

## Overview

As machine learning systems are deployed in high-stakes domains (healthcare, finance, hiring, criminal justice), the ability to understand *why* a model makes a decision becomes as important as the decision itself. **Transparency** refers to how openly a model's mechanism, training data, and limitations are disclosed. **Explainability** (or **interpretability**) refers to the degree to which a human can understand the cause of a model's prediction.

This notebook surveys the core concepts and the practical, post-hoc techniques used to explain model behavior, with from-scratch numpy implementations of the key algorithms.

---

## 1. Interpretable vs Explainable Models

- **Interpretable (transparent) models** are understandable by design. Their structure directly reveals how inputs map to outputs. Examples: **linear / logistic regression**, **decision trees** of limited depth, **rule lists**, and **generalized additive models (GAMs)**.
- **Explainable models** are typically **black-box** models (deep neural networks, gradient-boosted ensembles) whose internals are too complex to read directly. We attach **post-hoc** explanation methods on top of them to approximate or attribute their behavior.

A linear model is interpretable because the prediction is a transparent weighted sum:

$$ \hat{y} = w_0 + \sum_{i=1}^{d} w_i x_i $$

Each weight $w_i$ is the marginal contribution of feature $x_i$, holding others fixed.

---

## 2. Global vs Local Explanations

- **Global explanations** describe the model's overall behavior across the entire dataset (e.g. which features matter most on average). Methods: **permutation feature importance**, **Partial Dependence Plots (PDP)**, global **SHAP** summary plots.
- **Local explanations** describe a single prediction for one specific instance (e.g. why *this* loan was denied). Methods: **LIME**, local **SHAP** values, **Individual Conditional Expectation (ICE)** curves.

---

## 3. Post-hoc Explanation Methods

### 3.1 SHAP and Shapley Values

**SHAP** (SHapley Additive exPlanations) borrows the **Shapley value** from cooperative game theory. Each feature is a *player*; the *payout* is the model prediction. The Shapley value $\phi_i$ fairly distributes the prediction among features by averaging each feature's marginal contribution over all possible coalitions (subsets) $S$ of the other features:

$$ \phi_i = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|!\,(n - |S| - 1)!}{n!} \left[ v(S \cup \{i\}) - v(S) \right] $$

where $N$ is the set of all $n$ features, $v(S)$ is the model output using only the features in coalition $S$, and the combinatorial weight accounts for the number of orderings in which feature $i$ joins coalition $S$. SHAP values satisfy **local accuracy**, **missingness**, and **consistency**.

### 3.2 LIME

**LIME** (Local Interpretable Model-agnostic Explanations) explains a single prediction by fitting a simple, interpretable surrogate model (usually sparse linear) on perturbed samples drawn around the instance, weighted by proximity. It solves:

$$ \xi(x) = \arg\min_{g \in G} \; \mathcal{L}(f, g, \pi_x) + \Omega(g) $$

where $f$ is the black-box model, $g$ the surrogate, $\pi_x$ a locality kernel around $x$, and $\Omega(g)$ a complexity penalty.

### 3.3 Partial Dependence Plots (PDP) and ICE

A **Partial Dependence Plot** shows the marginal effect of one feature $x_S$ on the predicted outcome by averaging out all other features $x_C$:

$$ \hat{f}_S(x_S) = \frac{1}{n} \sum_{j=1}^{n} \hat{f}\big(x_S, x_C^{(j)}\big) $$

An **Individual Conditional Expectation (ICE)** curve is the same computation but kept *per instance* (one curve per row) instead of averaged, exposing heterogeneous effects that a PDP would wash out.

### 3.4 Grad-CAM

**Grad-CAM** (Gradient-weighted Class Activation Mapping) explains convolutional networks by using the gradients of a target class flowing into the last convolutional layer to produce a coarse localization heatmap highlighting the image regions most responsible for the prediction.

### 3.5 Attention Visualization

For transformer models, **attention visualization** inspects the attention weight matrices to show which tokens the model attends to. Attention is suggestive but not a faithful explanation by itself; it should be combined with attribution methods.

---

## 4. Documentation Artifacts

- **Model Cards** document a model's intended use, performance across subgroups, training data, ethical considerations, and limitations.
- **Datasheets for Datasets** document a dataset's motivation, composition, collection process, preprocessing, and recommended uses, mirroring electronics datasheets.

These artifacts support transparency even when the model itself is a black box.

---


In [1]:
# == Setup: synthetic data and a simple linear model trained in numpy ==
import numpy as np

rng = np.random.default_rng(42)

# Synthetic dataset: 4 features. True relationship is linear with known weights.
n_samples, n_features = 400, 4
X = rng.normal(0.0, 1.0, size=(n_samples, n_features))
true_w = np.array([3.0, -2.0, 0.0, 1.0])   # feature 2 is irrelevant
true_b = 0.5
noise = rng.normal(0.0, 0.5, size=n_samples)
y = X @ true_w + true_b + noise

# == Closed-form ordinary least squares (normal equations) ==
X_aug = np.hstack([np.ones((n_samples, 1)), X])      # prepend bias column
theta = np.linalg.lstsq(X_aug, y, rcond=None)[0]     # [b, w1, w2, w3, w4]
b_hat, w_hat = theta[0], theta[1:]

def model_predict(Xmat):
    """Black-box-style prediction interface for the explanation methods below."""
    return Xmat @ w_hat + b_hat

print('Learned bias :', round(float(b_hat), 3))
print('Learned weights:', np.round(w_hat, 3))
print('True weights   :', true_w)
# An interpretable model: each weight IS the global explanation.


Learned bias : 0.431
Learned weights: [ 2.989 -1.989 -0.03   0.992]
True weights   : [ 3. -2.  0.  1.]


In [2]:
# == Permutation feature importance from scratch (global explanation) ==
# Idea: shuffle one feature column, measure how much the error increases.
# A large increase means the model relied heavily on that feature.

def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))

def permutation_importance(predict_fn, X, y, n_repeats=10, seed=0):
    rng_local = np.random.default_rng(seed)
    baseline = mse(y, predict_fn(X))
    importances = np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        scores = []
        for _ in range(n_repeats):
            X_perm = X.copy()
            rng_local.shuffle(X_perm[:, j])      # break feature j's relationship
            scores.append(mse(y, predict_fn(X_perm)) - baseline)
        importances[j] = np.mean(scores)
    return importances

imp = permutation_importance(model_predict, X, y, n_repeats=20, seed=1)
for j, val in enumerate(imp):
    print(f'feature {j}: importance = {val:.3f}')
# Feature 2 (true weight 0) should show near-zero importance.


feature 0: importance = 16.766
feature 1: importance = 8.718
feature 2: importance = 0.002
feature 3: importance = 1.789


In [3]:
# == Exact Shapley values by enumerating all subsets (numpy) ==
# For a single instance x, we attribute the prediction to each feature.
# v(S) = model prediction when only features in S take x's values and the
# rest are replaced by a baseline (here the dataset mean = 'absent' feature).
from itertools import combinations
from math import factorial

baseline_x = X.mean(axis=0)   # reference 'absent' values

def coalition_value(S, x):
    """Prediction using features in set S from x, others from baseline."""
    z = baseline_x.copy()
    for i in S:
        z[i] = x[i]
    return float(model_predict(z.reshape(1, -1))[0])

def exact_shapley(x):
    n = len(x)
    features = list(range(n))
    phi = np.zeros(n)
    for i in features:
        others = [f for f in features if f != i]
        for r in range(len(others) + 1):
            for S in combinations(others, r):
                S = list(S)
                weight = factorial(len(S)) * factorial(n - len(S) - 1) / factorial(n)
                marginal = coalition_value(S + [i], x) - coalition_value(S, x)
                phi[i] += weight * marginal
    return phi

x_instance = X[0]
phi = exact_shapley(x_instance)
print('Instance:', np.round(x_instance, 3))
print('Shapley values:', np.round(phi, 3))
# Local accuracy check: baseline prediction + sum(phi) == full prediction.
recon = coalition_value([], x_instance) + phi.sum()
print('Reconstructed prediction:', round(recon, 3))
print('Actual model prediction :', round(float(model_predict(x_instance.reshape(1, -1))[0]), 3))


Instance: [ 0.305 -1.04   0.75   0.941]
Shapley values: [ 0.996  1.912 -0.023  0.926]
Reconstructed prediction: 4.321
Actual model prediction : 4.321


In [4]:
# == 1-D Partial Dependence (PDP) and ICE from scratch ==
# Vary one feature over a grid; for PDP average predictions over all rows,
# for ICE keep one curve per row.

def partial_dependence(predict_fn, X, feature_idx, grid_size=15):
    grid = np.linspace(X[:, feature_idx].min(), X[:, feature_idx].max(), grid_size)
    ice = np.zeros((X.shape[0], grid_size))
    for k, g in enumerate(grid):
        X_mod = X.copy()
        X_mod[:, feature_idx] = g            # force feature to grid value
        ice[:, k] = predict_fn(X_mod)        # one column per grid point
    pdp = ice.mean(axis=0)                    # average over instances
    return grid, pdp, ice

grid, pdp, ice = partial_dependence(model_predict, X, feature_idx=0, grid_size=10)
print('Grid values     :', np.round(grid, 2))
print('PDP (feature 0) :', np.round(pdp, 2))
# Approximate slope of the PDP recovers the linear weight for feature 0.
slope = np.polyfit(grid, pdp, 1)[0]
print('PDP slope ~ learned weight 0:', round(slope, 3), 'vs', round(float(w_hat[0]), 3))
print('ICE matrix shape (n_instances x grid):', ice.shape)


Grid values     : [-2.96 -2.31 -1.66 -1.01 -0.36  0.3   0.95  1.6   2.25  2.91]
PDP (feature 0) : [-8.27 -6.32 -4.37 -2.42 -0.47  1.48  3.43  5.38  7.33  9.28]
PDP slope ~ learned weight 0: 2.989 vs 2.989
ICE matrix shape (n_instances x grid): (400, 10)


In [5]:
# == Using production libraries (commented; install before running) ==
# pip install shap lime
#
# import shap
# explainer = shap.Explainer(model_predict, X)        # KernelExplainer-style
# shap_values = explainer(X[:50])
# shap.plots.beeswarm(shap_values)                     # global summary
# shap.plots.waterfall(shap_values[0])                 # local explanation
#
# from lime.lime_tabular import LimeTabularExplainer
# lime_exp = LimeTabularExplainer(X, mode='regression',
#                                 feature_names=['f0','f1','f2','f3'])
# exp = lime_exp.explain_instance(X[0], model_predict, num_features=4)
# print(exp.as_list())

# == Minimal Model Card template (printed) ==
model_card = '''
MODEL CARD
==========
Model name      : Synthetic Linear Regressor (demo)
Version         : 1.0
Date            : 2026-06-22
Intended use    : Educational demonstration of explanation methods.
Out-of-scope    : Any real decision-making or production deployment.
Architecture    : Ordinary least squares (4 features + bias).
Training data   : 400 synthetic samples, Gaussian features, known weights.
Metrics         : Train MSE reported per run; no separate test split here.
Ethical notes   : Synthetic data; no personal or sensitive attributes.
Limitations     : Linear; cannot capture non-linear interactions.
Caveats         : Feature 2 is irrelevant by construction (weight 0).
'''
print(model_card)



MODEL CARD
Model name      : Synthetic Linear Regressor (demo)
Version         : 1.0
Date            : 2026-06-22
Intended use    : Educational demonstration of explanation methods.
Out-of-scope    : Any real decision-making or production deployment.
Architecture    : Ordinary least squares (4 features + bias).
Training data   : 400 synthetic samples, Gaussian features, known weights.
Metrics         : Train MSE reported per run; no separate test split here.
Ethical notes   : Synthetic data; no personal or sensitive attributes.
Limitations     : Linear; cannot capture non-linear interactions.
Caveats         : Feature 2 is irrelevant by construction (weight 0).



## Additional Learning Resources

### Papers

- **A Unified Approach to Interpreting Model Predictions** (SHAP), Lundberg & Lee, 2017: https://arxiv.org/abs/1705.07874
- **Why Should I Trust You? Explaining the Predictions of Any Classifier** (LIME), Ribeiro et al., 2016: https://arxiv.org/abs/1602.04938

### Books & Courses

- **Interpretable Machine Learning** by Christoph Molnar (free online book): https://christophm.github.io/interpretable-ml-book/

### Tools & Libraries

- **SHAP**: https://github.com/shap/shap
- **LIME**: https://github.com/marcotcr/lime
- **InterpretML**: https://github.com/interpretml/interpret
- **Alibi**: https://github.com/SeldonIO/alibi
